# Ticket 12 Kaggle baseline

This notebook runs the exact locally validated candidate against the mounted Kaggle test layout. The delivery directory contains the immutable candidate, the separately hashed `kaggle.py` adapter, the copied delivery notebook, and the prepared `request.json`. The separately mounted approval input contains the owner approval for that exact request. It writes `submission.csv` and execution evidence to the working directory.

Internet is not required. Reports are not an inference input. The adapter verifies the request, approval, notebook hash, adapter hash, and pinned candidate environment before execution. A completed notebook is execution evidence, not platform acceptance or final selection.

In [ ]:
import time
notebook_started_at = time.perf_counter()

startup_setup_started = notebook_started_at
from pathlib import Path
import json
import sys

delivery_root = Path('/kaggle/input/rsna-knee-ticket12-delivery')
candidate_root = delivery_root / 'candidate'
code_root = delivery_root / 'code'
competition_input = Path('/kaggle/input/rsna-knee-abnormality-detection')
owner_approval_path = Path('/kaggle/input/rsna-knee-ticket12-approval/owner-approval.json')
delivery_notebook_path = delivery_root / 'ticket12_kaggle_baseline.ipynb'
delivery_adapter_path = code_root / 'rsna_knee' / 'kaggle.py'
request = json.loads((delivery_root / 'request.json').read_text(encoding='utf-8'))
sys.path.insert(0, str(code_root))
startup_setup_seconds = time.perf_counter() - startup_setup_started

imports_started = time.perf_counter()
import torch
from rsna_knee.kaggle import run_kaggle_baseline
imports_seconds = time.perf_counter() - imports_started

assert request['status'] == 'owner_approval_required'
assert request['candidate']['manifest_checksum']
assert request['delivery_notebook']['sha256']
assert request['delivery_adapter']['sha256']


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
result = run_kaggle_baseline(
    candidate_root,
    competition_input,
    Path('/kaggle/working/submission.csv'),
    evidence_path=Path('/kaggle/working/kaggle-execution.json'),
    private_evidence_path=Path('/kaggle/working/private-execution.json'),
    delivery_request_checksum=request['request_checksum'],
    delivery_notebook_sha256=request['delivery_notebook']['sha256'],
    delivery_adapter_sha256=request['delivery_adapter']['sha256'],
    delivery_request_root=delivery_root,
    owner_approval_path=owner_approval_path,
    delivery_notebook_path=delivery_notebook_path,
    delivery_adapter_path=delivery_adapter_path,
    notebook_stage_timings={
        'startup_setup': startup_setup_seconds,
        'imports': imports_seconds,
    },
    device=device,
    notebook_started_at=notebook_started_at,
)
print(json.dumps(result.evidence, indent=2, sort_keys=True))


## After the run

Keep the private execution file and `submission.csv` private. The public execution JSON records the exact candidate, adapter, notebook, pinned environment, resource measurements, workload, stage timings, projection, output checksum, and fallback accounting. It intentionally reports platform acceptance as pending; only after Kaggle accepts the output may the evidence be recorded with the ticket-12 verification command. Unknown hidden fallback information must remain unknown.